In [ ]:
!pip install qiskit qiskit-machine-learning qiskit-aer --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
!mkdir -p /content/data/processed
!unzip -q /content/drive/MyDrive/features_q.zip -d /content/data/processed
!ls /content/data/processed/features_q

In [ ]:
from pathlib import Path
from src.qcnn.train import TrainConfig, fit

cfg = TrainConfig(
    data_dir=Path("/content/data/processed/features_q"),
    n_qubits=8,
    n_blocks=1,
    measured_qubits=[0,1,2,3],
    n_classes=9,
    batch_size=8,
    epochs=15,
    lr=1e-3,
    eps=1e-10,
    seed=42,
)

model, history = fit(cfg)

In [ ]:
best_epoch = max(range(len(history["val"])), key=lambda i: history["val"][i]["acc"]) + 1

print("Best epoch:", best_epoch)
print("Train:", history["train"][best_epoch-1])
print("Val  :", history["val"][best_epoch-1])

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report

def qcnn_predict(model, X_np, batch_size=8):
    model.eval()
    dl = DataLoader(TensorDataset(torch.from_numpy(X_np.astype(np.float32))), batch_size=batch_size)
    probs_all = []
    with torch.no_grad():
        for (Xb,) in dl:
            probs16 = model(Xb)
            probs9 = probs16[:, :9]
            probs9 = torch.clamp(probs9, min=1e-10)
            probs9 = probs9 / probs9.sum(dim=1, keepdim=True)
            probs_all.append(probs9.cpu().numpy())
    probs_all = np.concatenate(probs_all, axis=0)
    y_pred = probs_all.argmax(axis=1)
    return y_pred, probs_all

DATA_DIR = Path("/content/data/processed/features_q")

X_test = np.load(DATA_DIR/"X_test.npy")
y_test = np.load(DATA_DIR/"y_test.npy")

y_pred, probs_test = qcnn_predict(model, X_test)

acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")
macro_prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
macro_rec  = recall_score(y_test, y_pred, average="macro", zero_division=0)

print("TEST METRICS")
print("Accuracy     :", acc)
print("Macro F1     :", macro_f1)
print("Weighted F1  :", weighted_f1)
print("Macro Prec   :", macro_prec)
print("Macro Recall :", macro_rec)

print("\nClassification Report")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

print("Confusion Matrix")
print(confusion_matrix(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6,5))
plt.imshow(cm)
plt.title("QCNN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()
plt.show()